**[🏠 Course Home](../README.md)** | **📖 [Conceptual](../conceptual/09_mcmc_mechanics_from_scratch.ipynb)** | **🤖 [AI-Native](09_mcmc_mechanics_from_scratch.ipynb)** | **🐍 [Python](../python/09_mcmc_mechanics_from_scratch.ipynb)** | **🔵 [R](../r/09_mcmc_mechanics_from_scratch.ipynb)** | **⬅️ [Prev: Ch 08](08_markov_chains_and_detailed_balance.ipynb)** | **➡️ [Next: Ch 10](10_mcmc_production_diagnostics.ipynb)**

*🧭 Part II: Numerical & Sampling Engines — Chapter 09 of 22*

---

# Chapter 9: The Metropolis Protocol — MCMC Mechanics from Scratch
### *Floating-Point Underflow, Log-Scale Decisions, and Step Tuning*

> **🧭 Course Context & Why This Module Matters in the AI Era:**
> In Chapter 8, we discovered that detailed balance allows a Markov chain to
> sample from an unnormalized discrete probability distribution. In this
> chapter, we generalize this principle to continuous parameters via the
> **Metropolis Algorithm**. The Metropolis algorithm is celebrated because it
> completely bypasses the intractable marginal likelihood denominator
> $P(\mathcal{D})$: the evidence constant cancels out in the acceptance ratio.
> However, when software engineers prompt an LLM to generate an MCMC sampler
> from scratch, the AI almost invariably writes likelihood evaluations in raw
> probability space. For realistic datasets ($N > 200$), multiplying raw
> probabilities causes immediate floating-point underflow to zero, producing
> `NaN` crashes or frozen walkers. This chapter establishes the robust,
> log-scale computational contract required for production-grade MCMC.

## Part 1: The Naive Prompt Trap & Forensic Post-Mortem

When developers ask an LLM to build a bespoke Markov Chain Monte Carlo sampler,
they usually describe the algorithm in textbook mathematical terms.

### The Naive Prompt
```text
Write a clean Python function implementing the Metropolis algorithm from
scratch to estimate the mean mu from a dataset of 300 response times. Use a
Normal(170, 15) prior and Normal(mu, 8.0) likelihood. Propose candidate steps
using a normal random walk.
```

### The Plausible-Sounding AI Hallucination
The LLM generates clean-looking, textbook-faithful Python code:
```python
import numpy as np
import scipy.stats as stats


def metropolis_sampler(data, n_samples=5000, start_mu=160.0, step_size=1.0):
  samples = [start_mu]
  current_mu = start_mu

  for _ in range(n_samples):
    # Propose new candidate
    prop_mu = current_mu + np.random.normal(0, step_size)

    # Compute raw prior and likelihood product
    p_curr_prior = stats.norm.pdf(current_mu, 170.0, 15.0)
    p_curr_lik = np.prod(stats.norm.pdf(data, current_mu, 8.0))
    p_curr = p_curr_prior * p_curr_lik

    p_prop_prior = stats.norm.pdf(prop_mu, 170.0, 15.0)
    p_prop_lik = np.prod(stats.norm.pdf(data, prop_mu, 8.0))
    p_prop = p_prop_prior * p_prop_lik

    # Acceptance probability
    alpha = min(1.0, p_prop / p_curr)

    if np.random.uniform(0, 1) < alpha:
      current_mu = prop_mu
    samples.append(current_mu)

  return np.array(samples)
```

```
                 THE FLOATING-POINT UNDERFLOW CATASTROPHE
                 
  Raw Likelihood Product (N = 300 data points):
  
    p(D | mu) = p(y_1 | mu) * p(y_2 | mu) * ... * p(y_300 | mu)
              ~ (0.05) * (0.05) * ... * (0.05)
              ~ (0.05)^300
              ~ 10^(-390)
              
  IEEE 754 Double Precision Subnormal Limit: ~ 10^(-324)
  
              +--------------------------------------------------+
              | Result: p_curr = 0.0,  p_prop = 0.0              |
              | Acceptance Ratio: alpha = 0.0 / 0.0 = NaN!       |
              | Sampler Fails: Walker Freezes or Crashes!        |
              +--------------------------------------------------+
```

---

### Forensic Post-Mortem: Why the AI Failed
The AI's code is mathematically textbook-correct on paper, but computationally
fatal in finite-precision hardware:

1. **Catastrophic IEEE 754 Underflow:**
   Standard 64-bit IEEE 754 floating-point numbers support a minimum positive
   subnormal number of $\approx 4.9 \times 10^{-324}$ and normalized minimum of
   $\approx 2.2 \times 10^{-308}$. For $N=300$ observations, each Gaussian
   density evaluation is roughly $\mathcal{O}(10^{-2})$ to $\mathcal{O}(0.1)$.
   Multiplying 300 probabilities yields an underflow to exact zero:
   $$\prod_{i=1}^{300} p(y_i \mid \mu) \to 0.0000000000000000$$
   Both `p_curr` and `p_prop` collapse to `0.0`. Their ratio `p_prop / p_curr`
   evaluates to `NaN`. In Python, comparisons `np.random.uniform(0, 1) < np.nan`
   always evaluate to `False`. The sampler silently freezes at `start_mu`
   forever!

2. **Log-Scale Arithmetic is Mandatory:**
   A calibrated probabilistic engineer works exclusively in **log-posterior**
   space. The product of probabilities transforms into a stable sum of
   log-densities:
   $$\log p(\theta \mid \mathcal{D}) = \log p(\theta) + \sum_{i=1}^N \log p(y_i \mid \theta) + \text{const}$$
   The acceptance ratio in log-space is simply the difference:
   $$\Delta \ell = \log p(\theta_{\text{prop}} \mid \mathcal{D}) - \log p(\theta_{\text{curr}} \mid \mathcal{D})$$
   $$\log \alpha = \min(0.0, \Delta \ell)$$
   The decision rule evaluates:
   $$\log(u) < \Delta \ell \quad \text{where } u \sim \text{Uniform}(0, 1)$$
   This formulation evaluates stably for $N=10^6$ observations without a single
   underflow error.

3. **Arbitrary Proposal Scale Pathology:**
   The AI picked an arbitrary `step_size = 1.0` without explaining that step
   size controls acceptance rate. If step size is too small ($0.05$), acceptance
   is $99\%$ but exploration is sluggish random walk diffusion; if too large
   ($20.0$), acceptance collapses to $0.5\%$.

## Part 2: Prompting as Probabilistic Programming — Declarative Specifications

To eliminate AI statistical hallucinations, we must specify the generative
physics, prior distributions, and decision loss rather than asking vague
questions. Depending on your workflow, you can specify models using three
progressively simpler patterns:

---

### Level 1: The Everyday 3-Sentence Micro-Spec *(Fastest for Chat)*
Copy and paste these 3 plain sentences into any LLM:

```markdown
1. Model: Non-conjugate posterior with N = 300 observations.
2. Numerical Invariant: Compute acceptance probability strictly in log-posterior
   space Delta_ell = ell(prop) - ell(curr) to prevent floating-point underflow.
3. Proposal Tuning: Tune proposal scale sigma_step to maintain an acceptance
   rate between 23% and 50%.
```

---

### Level 2: The Model Formula Shorthand *(Concise Probabilistic DSL)*
For quick prompt engineering with statistical models:

```text
Log-Accept: log(alpha) = min(0, log P(D|prop) + log P(prop) - log P(D|curr) - log P(curr))
Update:     Accept if log(u) < log(alpha), where u ~ Uniform(0, 1)
Tuning:     Adjust proposal scale to achieve target 44% acceptance
```

---

### Level 3: The Socratic Interviewer Meta-Prompt *(Zero Effort)*
If you do not want to write the spec yourself, force the AI to interview you:

> *"Act as an MCMC Compiler Architect. I am writing a Metropolis-Hastings sampler
for 300 observations. Ask me 3 questions about log-likelihood summation and
proposal scale tuning before writing the sampling loop."*

---

<details>
<summary><b>📋 View Full Production YAML Contract (For Automated CI/CD Pipelines & Autonomous Agents)</b></summary>

```yaml
# ==============================================================================
# PROBABILISTIC SPECIFICATION: METROPOLIS MCMC NUMERICAL STABILITY CONTRACT
# ==============================================================================
module: "ai/09_mcmc_mechanics_from_scratch"
target_parameter:
  name: "mu"
  domain: "(-infinity, +infinity)"

observables:
  sample_size: 300
  data_vector: "observed_heights_or_latencies"

prior_model:
  distribution: "Normal(loc=170.0, scale=15.0)"
  evaluation: "stats.norm.logpdf(mu, 170.0, 15.0)"

likelihood_model:
  distribution: "Normal(loc=mu, scale=8.0)"
  evaluation: "np.sum(stats.norm.logpdf(data, loc=mu, scale=8.0))"
  raw_probability_products_prohibited: true

computational_invariants:
  arithmetic_space: "log_posterior"
  acceptance_rule:
    log_delta: "log_joint(prop_mu) - log_joint(curr_mu)"
    decision_condition: "log(uniform(0, 1)) < log_delta"
  proposal_kernel:
    type: "Gaussian_random_walk"
    initial_step_size: 2.0
    target_acceptance_rate_1d: 0.44
    target_acceptance_window: "[0.25, 0.55]"
  warmup_contract:
    total_iterations: 6000
    warmup_discard_count: 1000
    retained_samples: 5000

decision_loss_contract:
  summary_metrics: ["mean", "std", "median", "hdi_95", "acceptance_rate"]
  numerical_health_assertion: "assert np.all(np.isfinite(samples))"
```

</details>


## Part 3: The Calibrated Agent Response & Telemetry

When an epistemic AI agent receives the Level 1 Micro-Spec from Part 2, it
rejects raw likelihood products and naive probability multiplications. Instead,
it executes an exact log-space Metropolis random walk with tuned proposal
scale, producing verified mathematical telemetry on the 50-observation height
dataset ($N=50, \bar{y} = 172.52, \sigma = 8.0$):

```text
[STEP 1: ANALYTICAL GROUND TRUTH BENCHMARK]
  Data Summary: N = 50, y_bar = 172.52, sigma_known = 8.0
  Prior: mu ~ Normal(mu_0 = 170.0, tau_0 = 15.0)
  Analytical Posterior Precision:
    tau_post^(-2) = (1 / 15^2) + 50 / (8^2) = 0.00444 + 0.78125 = 0.78569
    tau_post = 1 / sqrt(0.78569) = 1.1281 cm
  Analytical Posterior Mean:
    mu_post = tau_post^2 * (170 / 15^2 + 50 * 172.52 / 8^2)
    mu_post = 1.2727 * (0.7556 + 134.7812) = 172.505 cm

[STEP 2: RUNNING NAIVE LINEAR PRODUCT SAMPLER (COMPARISON)]
  Evaluating raw likelihood product:
    p_curr = np.prod(norm.pdf(heights, 170.0, 8.0)) = 1.48e-71
  For N = 300 data points:
    p_curr = np.prod(norm.pdf(data_300, 170.0, 8.0)) = 0.000000e+00 (UNDERFLOW!)
    Acceptance ratio: 0.0 / 0.0 = NaN. Execution aborted.

[STEP 3: RUNNING CALIBRATED LOG-SCALE METROPOLIS ENGINE]
  Total Draws: 6,000 (1,000 Warmup Discarded + 5,000 Retained)
  Proposal Step Size: sigma_step = 2.0
  Log-Posterior Function:
    log_joint(mu) = -0.5 * ((mu - 170) / 15)^2 - 0.5 * sum(((y - mu) / 8)^2)
  Initial Walker State: mu_0 = 150.0 (Far away from mode)
  
  Warmup Phase (Steps 0 - 1000):
    Burn-in drift from 150.0 to 172.4 completed in 65 steps.
    Warmup discarded completely.
    
  Production Phase (Steps 1001 - 6000):
    Accepted steps: 2,214 / 5,000
    Empirical Acceptance Rate: 44.28% (Target window: 25% - 55%, optimal ~44%)
    Autocorrelation Time: tau_int = 4.2 iterations
    Effective Sample Size: ESS = 5000 / (1 + 2 * 3.2) = 1,190 independent draws

[STEP 4: POSTERIOR MOMENT VERIFICATION AGAINST BENCHMARK]
  MCMC Sample Mean: 172.49 cm (Analytical Benchmark: 172.505 cm)
    Estimation Error: |172.49 - 172.505| = 0.015 cm
    Monte Carlo Error: MCSE = s / sqrt(ESS) = 1.13 / sqrt(1190) = 0.033 cm
    Benchmark Error < 1 * MCSE (PASSED Gate 2)
  MCMC 95% Credible Interval: [170.28, 174.71] cm
```

### ASCII Trace & Acceptance Telemetry Plot
```text
  Parameter mu (cm) vs. Iteration Step
   mu
    |
176 +          .  :  : . :.  . :  : . :.  . :  : . :.  (Stationary Mixing)
    |        : .  . :  : . :.  . :  : . :.  . :  : .
174 +       . : .  . :  : . :.  . :  : . :.  . :  :
    |      :  . : .  . :  : . :.  . :  : . :.  . :
172 +     :   . : .  . :  : . :.  . :  : . :.  . :   <- mu_post = 172.51
    |    /
160 +   /   <- Rapid Burn-in Drift (65 iterations)
    |  /
150 + *     <- Initial Start: mu_0 = 150.0 cm
    +-------+-------+-------+-------+-------+-------> Iterations
    0      500     1000    2000    3000    5000
    [--- Warmup ---][---------- Production Sampling ----------]
 
 Acceptance Rate: 44.28% | Autocorrelation Time tau: 4.2 | ESS: 1,190
```

### Decision Telemetry Table
| Metric | Naive Linear Sampler | Calibrated Log-Metropolis |
|---|---|---|
| **Likelihood Space** | Linear $\prod p(y_i \mid \mu)$ | Log-Sum $\sum \log p(y_i \mid \mu)$ |
| **Numerical Stability** | Underflows ($0.0 \to \text{NaN}$) | Machine Precision ($[-10^6, 0]$) |
| **Proposal Tuning** | Fixed step (trapped or random) | Tuned $\sigma_{\text{step}} = 2.0$ |
| **Acceptance Rate** | Undefined / $0.0\%$ | **$44.28\%$** (Optimal ~44%) |
| **Ground Truth Error** | Sampler crashes | **$0.015$ cm** ($< 1 \times \text{MCSE}$) |
| **Production Ready** | Fatal Failure | **Certified Clean** |

## Part 4: The 4 Mandatory Verification Gates (How to Audit the Agent)

The AI agent delivered a clean log-scale Metropolis sampler and reported
posterior draws that recover the analytical posterior mean $\mu_{\text{post}} =
172.505$ cm.

However, as an **Epistemic Architect**, you must never trust an MCMC sampler
without auditing it against the mathematical laws of Markov chains. How do you
know the sampler doesn't suffer from subtle floating-point cancellation,
asymmetric proposal bias, hidden state trapping, or fatal random walk
diffusion?

Before deploying MCMC-derived parameter estimates or letting an automated agent
tune physical systems, you audit the engine against **four mandatory
verification gates**:

```
              THE 4 MANDATORY AGENTIC VERIFICATION GATES
┌────────────────────────────────────────────────────────────────────┐
│ Gate 1: Prior Predictive Check                                     │
│ Sane physical domain bounds before seeing real data                │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Domain Sanity Passed]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 2: Simulation-Based Calibration (SBC)                         │
│ Algorithmic proof: recovers ground truth across 5,000 simulations  │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Algorithm Soundness Passed]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 3: Mathematical Invariant Conservation                        │
│ Detailed balance / R-hat < 1.01 / ESS > 400 / 0 divergences        │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [MCMC Invariants & Convergence Verified]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 4: Posterior Predictive Retrodiction & Loss                   │
│ Sane predictive fit to telemetry & expected loss decision audit    │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │
                                  ▼ [SAFE FOR PRODUCTION DEPLOYMENT]
```

---

### Gate 1: Prior Predictive Check (Domain Sanity & Zero Underflow)

* **The Production Risk (Why This Gate Exists)**:
  Naive AI code computes raw likelihood products $\prod_{i=1}^N p(y_i \mid
  \theta)$. In IEEE 754 double precision ($10^{-308}$ minimum normal),
  multiplying more than $50$ to $100$ probabilities causes underflow to `0.0`.
  The acceptance ratio evaluates to $\frac{0.0}{0.0} = \text{NaN}$. If
  unchecked, the sampler freezes on a single state or crashes silent production
  pipelines. Gate 1 guarantees finite log-scale evaluation and realistic domain
  bounds before touching empirical data.
* **First-Principles Mathematical Mechanism**:
  Every likelihood calculation must operate strictly in log-space:
  $$\log p(D \mid \mu) = \sum_{i=1}^N \log p(y_i \mid \mu)$$
  $$= -\frac{N}{2} \log(2\pi\sigma^2) - \frac{1}{2\sigma^2} \sum_{i=1}^N (y_i - \mu)^2$$
  Prior predictive simulation draws $\mu^{(s)} \sim \mathcal{N}(\mu_0,
  \tau_0^2)$ and evaluates whether prior predictions generate synthetic
  datasets $y_{\text{sim}} \sim \mathcal{N}(\mu^{(s)}, \sigma_0^2)$ spanning
  physically admissible human heights ($120 \text{ cm} \le y \le 225 \text{
  cm}$).
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Zero-Underflow Invariant: Static AST scan confirms no `np.prod` on
    likelihoods; all log-joint evaluations return finite real floats (`assert
    np.all(np.isfinite(log_post))`) (`[PASS]`).
  * Support Bounds Invariant: $99.9\%$ of prior predictive observations
    $y_{\text{sim}} \in [120.0, 225.0]$ cm (`[PASS]`).
  * Non-Degenerate Prior Variance: $\text{Var}_{\text{prior}}(\mu) = 225.0 > 0$
    (`[PASS]`).
  * Finite Trace Invariant: Exact count of `NaN` or `Inf` in chain trajectory
    is strictly $0$ (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 1 (Numerical Log-Scale & Prior Domain Sanity):
  Model: y ~ Normal(mu, sigma=8.0), mu ~ Normal(170, 15).
  1. Inspect sampler AST: assert no np.prod or raw norm.pdf calls.
  2. Simulate 10,000 prior predictive draws:
     mu^(s) ~ Normal(170, 15), y_sim ~ Normal(mu^(s), 8).
  3. Verify 99.9% of y_sim fall strictly in [120.0, 225.0] cm.
  4. Assert all log-posterior evaluations are finite floats (no NaN/Inf).
  Report [PASS] with prior predictive quantiles, or [FAIL] with violations.
  ```

---

### Gate 2: Simulation-Based Calibration (SBC) / Parameter Recovery

* **The Production Risk (Why This Gate Exists)**:
  An agent might write an asymmetric proposal $q(\theta^* \mid \theta) \neq
  q(\theta \mid \theta^*)$ without applying the Hastings correction factor
  $\frac{q(\theta \mid \theta^*)}{q(\theta^* \mid \theta)}$, or accidentally
  invert the log-acceptance ratio $\log \alpha = \ell(\text{curr}) -
  \ell(\text{prop})$. The sampler will quietly converge to an erroneous,
  inverted distribution while outputting plausible-looking numbers.
* **First-Principles Mathematical Mechanism**:
  SBC draws true parameters $\mu^* \sim \mathcal{N}(170, 15^2)$, generates
  synthetic data $y_{\text{fake}} \sim \mathcal{N}(\mu^*, 8^2)$ for $N=50$,
  executes the MCMC engine to draw $S=2{,}000$ posterior samples, and computes
  the rank statistic:
  $$r = \sum_{s=1}^S \mathbb{I}(\mu^{(s)} < \mu^*)$$
  Over $M = 1{,}000$ simulations, $r$ must be uniformly distributed on $\{0,
  1, \dots, S\}$.
  On a conjugate normal model, the posterior is analytically known:
  $$\tau_{\text{post}}^{-2} = \tau_0^{-2} + N \sigma^{-2} = \frac{1}{225} + \frac{50}{64} = 0.78569$$
  $$\tau_{\text{post}} = \frac{1}{\sqrt{0.78569}} \approx 1.1281 \text{ cm}$$
  $$\mu_{\text{post}} = \tau_{\text{post}}^2 \left(\frac{\mu_0}{\tau_0^2} + \frac{N \bar{y}}{\sigma^2}\right) \approx 172.505 \text{ cm}$$
  Empirical MCMC mean must match $\mu_{\text{post}}$ within $2 \times
  \text{MCSE} = 2 \times \frac{s_{\mu}}{\sqrt{\text{ESS}}}$.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Rank Uniformity Invariant: Kolmogorov-Smirnov test on normalized ranks
    against $\text{Uniform}(0, 1)$ yields $p_{\text{KS}} > 0.010$ (`[PASS]`).
  * Analytical Recovery Invariant:
    $$|\bar{\mu}_{\text{MCMC}} - \mu_{\text{post}}| \le 2 \times \frac{s_{\mu}}{\sqrt{\text{ESS}}} \quad (\text{[PASS]})$$
  * Posterior Scale Recovery: $|s_{\mu} - \tau_{\text{post}}| \le 0.050$ cm
    (`[PASS]`).
  * Recovery Bias Invariant: Mean recovery error across $1{,}000$ trials
    $|\frac{1}{M}\sum (\bar{\mu}_m - \mu_m^*)| \le 0.050$ cm (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 2 (Simulation-Based Calibration on Metropolis Engine):
  Execute 1,000 Monte Carlo recovery trials:
  For m = 1 to 1,000:
    1. Draw mu* ~ Normal(170, 15).
    2. Draw y_fake ~ Normal(mu*, 8) for N = 50.
    3. Run 2,000 Metropolis draws with log-space acceptance.
    4. Compute rank r_m = sum(mu^(s) < mu*).
  Perform Kolmogorov-Smirnov test on ranks against Uniform(0, 2000).
  Verify empirical mean recovers mu_post within 2 * MCSE.
  Report [PASS] with KS p-value, or [FAIL] with recovery bias.
  ```

---

### Gate 3: Mathematical Invariant Conservation (Detailed Balance & Tuning)

* **The Production Risk (Why This Gate Exists)**:
  Proposal variance $\sigma_{\text{step}}^2$ dictates exploration efficiency.
  If $\sigma_{\text{step}} = 0.05$ (too small), acceptance is $98\%$ but the
  chain diffuses by pure Brownian motion, taking millions of steps to explore
  the distribution (huge autocorrelation, $\text{ESS} < 50$). If
  $\sigma_{\text{step}} = 50.0$ (too large), acceptance drops to $2\%$,
  trapping the chain in place for thousands of iterations.
* **First-Principles Mathematical Mechanism**:
  Metropolis-Hastings detailed balance on continuous densities:
  $$p(\theta) q(\theta^* \mid \theta) \alpha(\theta \to \theta^*) = p(\theta^*) q(\theta \mid \theta^*) \alpha(\theta^* \to \theta)$$
  where the acceptance probability in log-space is:
  $$\log \alpha = \min(0.0, \Delta \ell) \le 0.0$$
  $$\Delta \ell = \log p(D \mid \theta^*) + \log p(\theta^*) - \log p(D \mid \theta) - \log p(\theta)$$
  For symmetric random walk proposals ($q(\theta^* \mid \theta) = q(\theta \mid
  \theta^*)$), proposal probabilities cancel identically.
  By the Gelman-Roberts-Gilks (1997) theorem, the optimal acceptance rate for a
  1D Gaussian random walk Metropolis sampler is $\approx 44\%$ (decaying to
  $\approx 23.4\%$ as dimension $D \to \infty$).
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Log-Acceptance Bound: $\log \alpha \le 0.0$ strictly for every step
    (`[PASS]`).
  * Production Acceptance Window: Retained post-warmup acceptance rate
    $\hat{\alpha} \in [0.25, 0.55]$ (`[PASS]`).
  * Autocorrelation Decay: Integrated autocorrelation time $\tau_{\text{int}}
    \le 25$ iterations (`[PASS]`).
  * Unique State Diversity: Unique parameter values in retained sample $\ge
    0.20 \times N_{\text{samples}}$ (`[PASS]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 3 (Metropolis Acceptance Calibration & Invariants):
  Given 5,000 retained Metropolis draws for mu:
  1. Verify log(alpha) <= 0.0 for 100% of proposed transitions.
  2. Compute empirical acceptance rate: assert 0.25 <= rate <= 0.55.
  3. Compute autocorrelation time tau_int: assert tau_int <= 25.
  4. Assert ESS >= 400 for production certification.
  Report [PASS] with acceptance rate and ESS, or [FAIL] with tuning violations.
  ```

---

### Gate 4: Posterior Predictive Retrodiction & Loss (Clearance Decision)

* **The Production Risk (Why This Gate Exists)**:
  Accepting MCMC draws without checking posterior predictive retrodiction risks
  deploying an ill-fitting model to downstream physical decisions. If an
  ergonomics team provisions height clearances for airplane cabins, office
  doorways, or vehicle cockpits based on posterior mean alone, ignoring
  parameter uncertainty and individual variance leads to equipment where tall
  users hit their heads. Under-sizing carries severe asymmetric cost
  ($C_{\text{injury}} = 20 \times C_{\text{headroom}}$).
* **First-Principles Mathematical Mechanism**:
  1. **Posterior Predictive Distribution**:
     $$\tilde{y} \mid D \sim \mathcal{N}\left(\mu_{\text{post}}, \sigma^2 + \tau_{\text{post}}^2\right)$$
     $$= \mathcal{N}(172.505, 8.0^2 + 1.128^2) = \mathcal{N}(172.505, 65.27)$$
     Simulate replicated datasets $\tilde{y}^{(s)} \sim \mathcal{N}(\mu^{(s)},
     \sigma^2)$ for each posterior sample $\mu^{(s)}$.
  2. **Predictive Retrodiction Check**:
     Compute the retrodictive p-value for the sample mean:
     $$p_{\text{retro}} = P(\bar{\tilde{y}} \ge \bar{y}_{\text{obs}} \mid D) \approx 0.505$$
     Because $0.10 \le p_{\text{retro}} \le 0.90$, the model retrodicts the
     observed sample center cleanly.
  3. **Asymmetric Decision Loss on Clearance**:
     Let clearance height be $H^*$. Cost of under-sizing is $C_{\text{under}} =
     20$ if individual height $y > H^*$; cost of excess clearance is
     $C_{\text{over}} = 1$.
     The critical percentile threshold is:
     $$q^* = \frac{C_{\text{under}}}{C_{\text{under}} + C_{\text{over}}} = \frac{20}{21} \approx 0.9524 \quad (95.24\%)$$
     The Bayes-optimal clearance height is the $95.24\%$ quantile of the
     posterior predictive distribution:
     $$H^* = \mu_{\text{post}} + z_{0.9524} \sqrt{\sigma^2 + \tau_{\text{post}}^2}$$
     $$= 172.505 + 1.668 \times 8.079 \approx 185.98 \text{ cm}$$
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Retrodictive Mean Coverage: Observed mean $\bar{y} = 172.52$ falls inside
    the central $95\%$ predictive interval of means $[170.2, 174.8]$ cm
    (`[PASS]`).
  * Retrodictive P-Value: $0.10 \le p_{\text{retro}}(\text{mean}) \le 0.90$
    (`[PASS]`).
  * Bayes-Optimal Clearance Decision: Clearance height $H^* = 186.0 \pm 0.5$ cm
    (`[PASS]`; deploying posterior mean $172.5$ cm triggers immediate `[FAIL]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 4 (Posterior Predictive Checks & Asymmetric Loss Decision):
  Using 5,000 posterior draws mu^(s) ~ p(mu | D):
  1. Generate 5,000 replicated datasets y_rep ~ Normal(mu^(s), 8.0).
  2. Verify observed sample mean falls within 95% predictive interval.
  3. Compute retrodictive p-value for mean: assert 0.10 <= p_val <= 0.90.
  4. Minimize expected asymmetric loss with C_under / C_over = 20:
     find optimal quantile q* = 20 / 21 = 95.24%.
  Assert optimal clearance H* == 186.0 cm. Report [PASS] or [FAIL].
  ```

## Part 5: Naive AI Prompt vs. Calibrated Epistemic Specification

| Dimension | Naive AI Prompt | Calibrated Epistemic Specification |
|---|---|---|
| **Likelihood Arithmetic** | Direct multiplication `np.prod(pdf)` (underflows to 0 for $N>150$) | Log-space summation `np.sum(logpdf)` (numerically stable for all $N$) |
| **Acceptance Ratio** | Computes ratio in linear space $P^*/P$, causing `0.0/0.0 = NaN` | Computes difference $\Delta \ell = \ell^* - \ell$, evaluating $\log u < \Delta \ell$ |
| **Proposal Calibration** | Uses unprincipled static step sizes; ignores efficiency | Targets optimal acceptance ($44\%$ for 1D, $23.4\%$ for multi-D) |
| **Warmup Discard** | Frequently pools initial transient steps into posterior | Strictly discards pre-convergence warmup draws ($1000$ steps) |
| **Verification Gates** | None; runs script and reports whatever numbers finish | 4 gates: Underflow check, Log-bounds, Acceptance calibration, Benchmark |
| **Production Risk** | Silent sampler freeze or `NaN` crashes in CI/CD | Certified stable, automated convergence monitoring |

## Part 6: 🎯 Check Your Understanding — 3-Tier "Audit the AI" Exercises

### 🟢 Tier 1: Spot the AI Hallucination
An engineer asks an AI model to troubleshoot a custom MCMC sampler:
> *"I ran my Metropolis sampler on $N=500$ server response times. The walker
> moves for the first 3 steps, then freezes completely at $\mu = 168.4$ for the
> remaining 9,997 steps. The AI responds: 'Your model has converged to the exact
> global maximum with 100% confidence. Because the posterior mode has been
> located, the acceptance probability for any other value is zero, so the
> sampler correctly remains at the exact summit.' "*

Explain why the AI's explanation is completely fabricated, and identify the true
computational failure.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Forensic Audit:**
The AI has hallucinated that an MCMC sampler is an optimizer that stops moving
upon finding a mode!
1. **MCMC Never Freezes at Equilibrium:** A valid Markov chain never stops
   wandering; it must continue exploring the posterior distribution in
   accordance with its density.
2. **True Cause (Floating-Point Underflow):** The sampler used raw probability
   multiplication `np.prod`. On step 4, the current position achieved a
   likelihood small enough that `np.prod` underflowed to exact `0.0`. Then
   `p_curr = 0.0`. In subsequent steps, any proposed candidate also underflows
   to `p_prop = 0.0`, resulting in `alpha = 0.0 / 0.0 = NaN`. The comparison
   `uniform < NaN` evaluates to `False`, rejecting all proposals indefinitely.
</details>

---

### 🟡 Tier 2: Refactor the Flawed Specification
Below is an uncalibrated agent prompt to implement a Metropolis-Hastings
sampler with an asymmetric proposal distribution:
$$q(\theta^* \mid \theta) = \text{LogNormal}(\mu=\log \theta, \sigma=0.5)$$

```text
Write a Metropolis sampler for a positive scale parameter sigma > 0. At each
step, propose candidate sigma_prop ~ LogNormal(mean=log(sigma_curr), sigma=0.5).
Calculate the log-posterior difference delta_log = log_post(sigma_prop) -
log_post(sigma_curr) and accept if log(uniform(0, 1)) < delta_log.
```

Explain why this sampler violates detailed balance, and rewrite the prompt into
a calibrated declarative specification that includes the Hastings correction
term.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Why it fails:**
Because the Log-Normal proposal is asymmetric, proposing $\theta^* \to \theta$
does not have the same probability density as proposing $\theta \to \theta^*$:
$$q(\theta^* \mid \theta) \neq q(\theta \mid \theta^*)$$
Omitting the Hastings correction factor:
$$\log q(\theta_{\text{curr}} \mid \theta_{\text{prop}}) - \log q(\theta_{\text{prop}} \mid \theta_{\text{curr}})$$
violates detailed balance, biasing samples toward larger values.

**Calibrated Declarative Specification:**
```yaml
module: "calibrated_metropolis_hastings_asymmetric"
target_parameter:
  name: "sigma"
  domain: "(0.0, +infinity)"

proposal_kernel:
  family: "LogNormal"
  distribution: "theta_prop ~ LogNormal(meanlog=log(theta_curr), sdlog=tau)"
  hastings_correction_log: "log(theta_prop) - log(theta_curr)"

acceptance_rule:
  log_alpha: "delta_log_joint + hastings_correction_log"
  decision: "log(uniform(0, 1)) < min(0.0, log_alpha)"

validation:
  assert_detailed_balance: "verify Hastings ratio cancels proposal asymmetry"
```
</details>

---

### 🔴 Tier 3: Architect an Autonomous Agent Contract
Architect an autonomous agent contract in pseudo-code that implements an
adaptive proposal tuner during warmup: using Robbins-Monro stochastic
approximation, it dynamically adjusts proposal scale $\sigma_{\text{step}}$ to
achieve target acceptance $0.44$, freezes adaptation strictly at step $W$, and
verifies that post-warmup acceptance stays within $[0.35, 0.55]$.

<details>
<summary><b>Click here to view the solution...</b></summary>

**Autonomous Contract Architecture:**
```python
def calibrated_adaptive_metropolis(
    log_joint_fn, init_theta, n_draws=6000, n_warmup=1000
):
  theta = float(init_theta)
  log_sigma_step = np.log(1.0)  # Initial proposal scale
  target_accept = 0.44
  samples = []
  accepted_post_warmup = 0

  curr_log_p = log_joint_fn(theta)

  for t in range(1, n_draws + 1):
    sigma_step = np.exp(log_sigma_step)
    prop_theta = theta + np.random.normal(0, sigma_step)
    prop_log_p = log_joint_fn(prop_theta)

    # Log-space Metropolis decision
    delta_log = prop_log_p - curr_log_p
    log_u = np.log(np.random.uniform(1e-12, 1.0))
    accepted = log_u < min(0.0, delta_log)

    if accepted:
      theta = prop_theta
      curr_log_p = prop_log_p

    # Robbins-Monro adaptation during warmup ONLY
    if t <= n_warmup:
      gamma_t = 1.0 / (t**0.7)  # Decreasing step size
      alpha_t = min(1.0, np.exp(delta_log)) if np.isfinite(delta_log) else 0.0
      log_sigma_step += gamma_t * (alpha_t - target_accept)
    else:
      # Production phase: Proposal scale FROZEN
      samples.append(theta)
      if accepted:
        accepted_post_warmup += 1

  # Post-sampling verification
  production_draws = n_draws - n_warmup
  emp_accept_rate = accepted_post_warmup / production_draws

  assert (
      0.30 <= emp_accept_rate <= 0.60
  ), f'Acceptance rate {emp_accept_rate:.3f} outside valid window'

  return {
      'samples': np.array(samples),
      'final_step_size': float(np.exp(log_sigma_step)),
      'production_acceptance_rate': float(emp_accept_rate),
      'warmup_discarded': n_warmup,
  }
```
</details>

## Key Takeaways & Course Navigation

### 🎯 Key Takeaways:
1. **Never Multiply Linear Probabilities:** Likelihood products underflow
   to zero for sample sizes as small as $N=150$. All Bayesian computation must
   occur on the log-posterior scale.
2. **Acceptance Difference Rule:** In log-space, the acceptance ratio is a
   simple subtraction: $\Delta \ell = \ell^* - \ell$. The acceptance decision
   evaluates $\log u < \Delta \ell$.
3. **The Acceptance Tradeoff:** Aim for $\approx 44\%$ acceptance in 1D models
   and $\approx 23.4\%$ in multi-dimensional models. Tiny steps cause sluggish
   random-walk diffusion; huge steps cause frequent rejections.
4. **Freeze Adaptation Before Production:** If you tune proposal step size, you
   must freeze it during the retained sampling phase. Adapting the kernel
   during production breaks the Markov property and destroys the stationary
   distribution.

---

**[🏠 Course Home](../README.md)** | **📖 [Conceptual](../conceptual/09_mcmc_mechanics_from_scratch.ipynb)** | **🤖 [AI-Native](09_mcmc_mechanics_from_scratch.ipynb)** | **🐍 [Python](../python/09_mcmc_mechanics_from_scratch.ipynb)** | **🔵 [R](../r/09_mcmc_mechanics_from_scratch.ipynb)** | **⬅️ [Prev: Ch 08](08_markov_chains_and_detailed_balance.ipynb)** | **➡️ [Next: Ch 10](10_mcmc_production_diagnostics.ipynb)**